#### Connexion à la base de données PostgreSQL

In [15]:
import os
import pandas as pd
from dotenv import load_dotenv
from nutriscope import config
from pathlib import Path
from sqlalchemy import create_engine, text

load_dotenv()

url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@localhost:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

engine = create_engine(url)

with engine.connect() as connection:
    print(connection.execute(text('SELECT 1')).scalar())

1


#### Créations des tables

In [16]:
schema_sql = Path('../sql/schema.sql').read_text(encoding='utf-8')

with engine.begin() as connection:
    connection.execute(text(schema_sql))

##### Chargement des données

In [17]:
CSV_GZ = Path("..") / "data" / "france" / "food_france.csv.gz"

columns = config.COLUMNS

dataframe = pd.read_csv(
    CSV_GZ,
    sep="\t",
    nrows=5000,
    usecols=columns,
    dtype={"code": "string"},
    low_memory=False
)

dataframe.head()

,code,product_name,brands,categories,categories_tags,labels,countries,countries_tags,allergens,nutriscore_score,...,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,fruits-vegetables-legumes_100g
0,0000101209159,Véritable pâte à tartiner noisettes chocolat noir,Bovetti,NaN,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",NaN,NaN,en:france,NaN,25.0,...,2524.00,48.0,10.00,36.00,32.0,NaN,8.00,0.01,0.004,0.0
1,0000130008136,Escalope de dinde,NaN,"Viandes et dérivés, Viandes, Volailles, Dinde ...","en:meats-and-their-products,en:meats,en:poultr...",NaN,France,en:france,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0000140323687,Madeleine Framboise,NaN,"Snacks, Snacks sucrés, Biscuits et gâteaux, Gâ...","en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,France,en:france,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0000141013129,Croissants margarine,NaN,"Snacks,Snacks sucrés,en:Sweet pastries and pie...","en:snacks,en:sweet-snacks,en:sweet-pastries-an...",NaN,France,en:france,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0000171812457,Glaces vegetales des alpes,NaN,NaN,NaN,"Sans gluten, Sans lactose",France,en:france,NaN,NaN,...,839.01,9.2,6.21,27.11,26.6,NaN,2.22,0.10,0.040,NaN


#### Renommage des colonnes pour matcher avec la base de données

In [10]:
dataframe.columns = dataframe.columns.str.replace('-', '_')
dataframe.head()

,code,product_name,brands,categories,categories_tags,labels,countries,countries_tags,allergens,nutriscore_score,...,energy_100g,fat_100g,saturated_fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,fruits_vegetables_legumes_100g
0,0000101209159,Véritable pâte à tartiner noisettes chocolat noir,Bovetti,NaN,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",NaN,NaN,en:france,NaN,25.0,...,2524.00,48.0,10.00,36.00,32.0,NaN,8.00,0.01,0.004,0.0
1,0000130008136,Escalope de dinde,NaN,"Viandes et dérivés, Viandes, Volailles, Dinde ...","en:meats-and-their-products,en:meats,en:poultr...",NaN,France,en:france,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0000140323687,Madeleine Framboise,NaN,"Snacks, Snacks sucrés, Biscuits et gâteaux, Gâ...","en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,France,en:france,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0000141013129,Croissants margarine,NaN,"Snacks,Snacks sucrés,en:Sweet pastries and pie...","en:snacks,en:sweet-snacks,en:sweet-pastries-an...",NaN,France,en:france,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0000171812457,Glaces vegetales des alpes,NaN,NaN,NaN,"Sans gluten, Sans lactose",France,en:france,NaN,NaN,...,839.01,9.2,6.21,27.11,26.6,NaN,2.22,0.10,0.040,NaN
